# **Bronze Layer - Ingestion**

In [0]:
# importing libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# loading customer data 
df_customer = spark.read.format('csv')\
        .option('header', True)\
        .option('inferSchema', True)\
        .load('/Volumes/databricks_pyspark_dbt_project_01/source_files/source_files/customers_data/customers.csv')


In [0]:
# displaying customer data - only 10 records
df_customer.limit(100).display()

In [0]:
# storing schema 
schema_customer = df_customer.schema
schema_customer

### Bronze Layer Ingestion -> By Stream Processing -> By STATIC FORMAT (read & write each file manually)

In [0]:
# Spark Structure Streaming
# reading data
df_customer_stream = spark.readStream.format('csv')\
                            .option('header', True)\
                            .schema(schema_customer)\
                            .load('/Volumes/databricks_pyspark_dbt_project_01/source_files/source_files/customers_data')

# writing data 
df_customer_stream.writeStream.format("delta")\
                    .outputMode('append')\
                    .option('checkpointLocation', '/Volumes/databricks_pyspark_dbt_project_01/bronze_schema/checkpoint_locations/customers_data')\
                    .trigger(availableNow = True)\
                    .toTable('databricks_pyspark_dbt_project_01.bronze_schema.customers_data')

### Bronze Layer Ingestion -> By Stream Processing -> By DYNAMIC FORMAT (read & write each file automatically)

In [0]:
# create entities 
entities = ['customers_data', 'drivers_data', 'locations_data', 'payments_data', 'trips_data', 'vehicles_data']

for entity in entities:

        # read data by BATCH WAY
        df_batch = spark.read\
                        .format('csv')\
                        .option('header', True)\
                        .option('inferSchema', True)\
                        .load(f"/Volumes/databricks_pyspark_dbt_project_01/source_files/source_files/{entity}/")
        
        # grabbing all the schemas
        entity_schema = df_batch.schema

        # read data by STREAM WAY
        df_stream = spark.readStream\
                        .format('csv')\
                        .option('header', True)\
                        .schema(entity_schema)\
                        .load(f"/Volumes/databricks_pyspark_dbt_project_01/source_files/source_files/{entity}/")

        # write data by STREAM WAY
        df_stream.writeStream\
                        .format('delta')\
                        .outputMode('append')\
                        .option('checkpointLocation', f'/Volumes/databricks_pyspark_dbt_project_01/bronze_schema/checkpoint_locations/{entity}')\
                        .trigger(availableNow = True)\
                        .toTable(f'databricks_pyspark_dbt_project_01.bronze_schema.{entity}')
